In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from scipy import stats

# Target Paths
PROCESSED_DIR = "***/data/processed"
FIGURES_DIR = "***/reports/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Plotting Configuration
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

def run_anomaly_and_trend_analysis():
    global_df = pd.read_csv(os.path.join(PROCESSED_DIR, "climate_energy_global_merged.csv"))

    print("==================================================")
    print("🔬 ANOMALY DETECTION & ACCELERATION ANALYSIS")
    print("==================================================")

    # 1. Isolation Forest Anomaly Detection
    global_df["temp_diff"] = global_df["temp_anomaly_global"].diff().fillna(0)
    features = global_df[["temp_anomaly_global", "temp_diff"]]
    
    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    global_df["anomaly_label"] = iso_forest.fit_predict(features)
    anomalies = global_df[global_df["anomaly_label"] == -1]

    print(f"Detected {len(anomalies)} statistical climate anomaly years via Isolation Forest:")
    print(anomalies[["year", "temp_anomaly_global", "temp_diff"]].to_string(index=False))

    # 2. Acceleration Analysis (Slope comparison pre/post 1975)
    pre_1975 = global_df[global_df["year"] < 1975]
    post_1975 = global_df[global_df["year"] >= 1975]

    res_pre = stats.linregress(pre_1975["year"], pre_1975["temp_anomaly_global"])
    res_post = stats.linregress(post_1975["year"], post_1975["temp_anomaly_global"])

    slope_pre, intercept_pre, r_pre, p_pre = res_pre.slope, res_pre.intercept, res_pre.rvalue, res_pre.pvalue
    slope_post, intercept_post, r_post, p_post = res_post.slope, res_post.intercept, res_post.rvalue, res_post.pvalue

    print(f"\nWarming Acceleration Breakdown:")
    print(f"  Pre-1975 Rate  : {slope_pre:.5f} °C/year (R² = {r_pre**2:.3f}, p = {p_pre:.4e})")
    print(f"  Post-1975 Rate : {slope_post:.5f} °C/year (R² = {r_post**2:.3f}, p = {p_post:.4e})")
    print(f"  Acceleration Multiplier : {slope_post / slope_pre:.2f}x faster post-1975")

    # ==================================================
    # FIGURE GENERATION
    # ==================================================

    # Figure 6: Isolation Forest Anomaly Detection
    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)
    ax.plot(global_df["year"], global_df["temp_anomaly_global"], color="#1f77b4", linewidth=1.5, label="Temperature Anomaly (°C)", alpha=0.8)
    
    ax.scatter(
        anomalies["year"], anomalies["temp_anomaly_global"], 
        color="#d62728", s=50, zorder=5, label="Statistical Anomaly (Isolation Forest)", edgecolor="black"
    )

    ax.set_title("Isolation Forest Outlier & Extreme Event Detection (1880–2024)", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Global Temperature Anomaly (°C)")
    ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#cccccc", framealpha=1.0)
    plt.tight_layout()
    fig6_path = os.path.join(FIGURES_DIR, "fig6_anomaly_detection.png")
    plt.savefig(fig6_path)
    plt.close()
    print(f"\nSaved: {fig6_path}")

    # Figure 7: Structural Break / Pre vs Post 1975 Acceleration Comparison
    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)
    ax.scatter(global_df["year"], global_df["temp_anomaly_global"], color="gray", alpha=0.5, s=15, label="Observed Annual Mean")
    
    ax.plot(pre_1975["year"], slope_pre * pre_1975["year"] + intercept_pre, 
            color="#1f77b4", linewidth=2.5, label=f"1880–1974 Trend ({slope_pre*10:.3f} °C/decade)")
    
    ax.plot(post_1975["year"], slope_post * post_1975["year"] + intercept_post, 
            color="#d62728", linewidth=2.5, label=f"1975–2024 Trend ({slope_post*10:.3f} °C/decade)")

    ax.axvline(1975, color="black", linestyle="--", alpha=0.7, label="Structural Regime Shift (1975)")
    ax.set_title("Structural Shift in Thermal Drift Trajectory (Pre-1975 vs Post-1975)", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Global Temperature Anomaly (°C)")
    ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#cccccc", framealpha=1.0)
    plt.tight_layout()
    fig7_path = os.path.join(FIGURES_DIR, "fig7_regime_shift_acceleration.png")
    plt.savefig(fig7_path)
    plt.close()
    print(f"Saved: {fig7_path}")

if __name__ == "__main__":
    run_anomaly_and_trend_analysis()

🔬 ANOMALY DETECTION & ACCELERATION ANALYSIS
Detected 8 statistical climate anomaly years via Isolation Forest:
 year  temp_anomaly_global  temp_diff
 1890                -0.36      -0.25
 1957                 0.05       0.24
 1964                -0.20      -0.25
 1977                 0.18       0.28
 1999                 0.38      -0.23
 2021                 0.85      -0.16
 2023                 1.17       0.28
 2024                 1.28       0.11

Warming Acceleration Breakdown:
  Pre-1975 Rate  : 0.00365 °C/year (R² = 0.394, p = 1.0297e-11)
  Post-1975 Rate : 0.02042 °C/year (R² = 0.891, p = 8.7437e-25)
  Acceleration Multiplier : 5.59x faster post-1975

Saved: /Users/maharhandi/Desktop/Maha/DATA_ANALYST/Portfolio_Tracking/data_analysis_portfolio/project-3/reports/figures/fig6_anomaly_detection.png
Saved: /Users/maharhandi/Desktop/Maha/DATA_ANALYST/Portfolio_Tracking/data_analysis_portfolio/project-3/reports/figures/fig7_regime_shift_acceleration.png
